In [ ]:
import sys
import os.path as osp

import pandas as pd
import numpy as np

from combat.pycombat import pycombat  # pip install combat

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import *
from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import umap
import seaborn as sns  # For h-clust mostly
import plotly.express as px
import plotly.graph_objects as go  # For heatmaps

In [ ]:
# To make plotly fig show in notebook
import plotly.io as pio
pio.renderers.default = "notebook"

## Functions

In [ ]:
def plot_df(a_df, color_selected):
    # Plot df resulting from t-SNE or Isomap
    fig = px.scatter(
            a_df,
            x='compon0',
            y='compon1',
            hover_data=[a_df.index],
            color=color_selected
    )
    return fig

# MAIN

In [ ]:
# Read filtered TSV (only non fully '0' rows)
X = pd.read_csv("counts_norm.tsv", sep="\t", index_col=0)

# Compute epiSize (= size of epiSign):
# MEMO: Cannot compute epiSize with 'full values' (almost NO '0' values)
#epiSize = {}
#for epiSign in [x for x in X.columns if x not in ('coord')]:
#    epiSize[epiSign] = sum(X[epiSign] > 0)  # Catch 100% methyl

# If detect NaN or null values -> stop here:
cols_with_na = {}
for a_col in X.columns:
    nb_na = sum(X[a_col].isna())
    if nb_na > 0:
        cols_with_na[a_col] = nb_na
assert not cols_with_na, f"Some cols have missing values: {cols_with_na}"

## Pre-processing

### Batch-effect correction using (py)combat

In [ ]:
# Use file describing batches (1 by row)

datasets = []
with open('batches.tsv') as batches_file:
    for batch_line in batches_file:
        datasets.append(batch_line.rstrip('\n').split('\t'))

batch = []
for j in range(len(datasets)):
    batch.extend([j for _ in range(len(datasets[j]))])

# Then run (py)combat:
#X_corrected = pycombat(X, batch)

### Transpose + normalize

In [ ]:
USE_COMBAT = False
USE_NORMALIZED = False
EPISIGN_OF_INTEREST = ['HG002_combined','RMNS','Kleefstra','Kabuki','barcode04_combined']
# ONLY if 'full data from publi':
#EPISIGN_OF_INTEREST += ['EPI_01', 'EPI_02', 'EPI_08','EPI_19'] # 4 kab samples

# MirinFMF project
# Possible F samples:
#EPISIGN_OF_INTEREST = ["8ZZUUQM","CSG246672","CSG247928","CSG248017","CSG248018","CSG251485","CSG252095","CSG253228","CSG253525","CSG254002","CSG256433","CSG256435","E19WLVU","LQXGE7H","NL0500099","NL0800233","NL0800248","NL0800249","NL0800270","OD_2024101","PH5SJZD","PID_160383","PID_160392","PID_160401","PID_160403","PID_170302","PID_170308","PID_170365","PID_170366","R4XUJV2","R6CYPDM","RT8DFEP_FK2UZYZ","UAKAPLJ","V-87-DZ_16_10_n"]

# RNAseq for NFKB sign:
# Abritrary take 1st and 9th samples
EPISIGN_OF_INTEREST = X.columns[[0,10]]

# If we know true clusters:
TRUE_CLUST_FILE = "known_clusters.tsv"
if osp.isfile(TRUE_CLUST_FILE):
    known_clusters = pd.read_csv(TRUE_CLUST_FILE, sep="\t")
    # WARN: Order of samples should be the same as 'counts'
    assert list(known_clusters['sample']) == list(X.columns)
    EPISIGN_OF_INTEREST = known_clusters.cluster

# Transpose (required):
X_t = X.T
if USE_COMBAT:
    X_t = X_corrected.T

# Remove 2nd row = size of epiSign then normalize
to_norm = X_t
if 'epiSize' in X_t.columns:
    to_norm = X_t.drop('epiSize', axis=1)
print(to_norm[to_norm.columns[0:3]].head())

# WARN: If norm, should be AFTER combat
to_PCA = to_norm
if USE_NORMALIZED:
    to_PCA = pd.DataFrame(StandardScaler().fit_transform(to_norm), columns=to_norm.columns, index=to_norm.index)

# Write corrected file:
for sample in ['barcode04_combined', 'HG002_combined']:
    #to_PCA.T[sample].to_csv(f"{sample}_corrected.tsv", sep="\t")
    print(f" > Wrote: {sample}_corrected.tsv")

## Vizu / dimension reduction

### PCA

#### Selection du nombre de compon

In [ ]:
# Select best number of components

pca = PCA(random_state=42)
pca.fit(to_PCA.to_numpy())

cumvar = pca.explained_variance_ratio_.cumsum()

fig = px.line(cumvar)
fig.show()

NB_COMPON = np.argmax(cumvar >= 0.80)+1 + 2  # Add '1' to have at least 3 compon even if best is '1'
print(f"Number of PCA components for '80%' variance explained: {NB_COMPON}")

In [ ]:
# Run PCA:

pca = PCA(
    n_components=NB_COMPON,
    random_state=42
)
pcs = pca.fit_transform(to_PCA.to_numpy())

print(f"Trustworthiness of the low-dimensional embedding for PCA: {trustworthiness(to_PCA, pcs, n_neighbors=2)}")
print( "Variance expliquée PC1 :", round( pca.explained_variance_ratio_[0] * 100, 2 ), "%" )
print( "Variance expliquée PC2 :", round( pca.explained_variance_ratio_[1] * 100, 2 ), "%" )
print( "Total variance explained:", sum(pca.explained_variance_ratio_)*100)

# Make a dict with '% variance explained' for each component:
dict_compon = {'compon'+str(i) : str(round(pca.explained_variance_ratio_[i]*100,4)) for i in range(NB_COMPON)}

#### Plots

In [ ]:
# Put results into a dataframe
# WARN: bellow 'dict.keys' assume orderedDict
pcs_df = pd.DataFrame(
    pcs,
    index=X.columns,
    columns=dict_compon.keys()
)
#print(pcs_df.loc[EPISIGN_OF_INTEREST])

In [ ]:
# Plot 1st and 2nd compon from PCA
color_selected = [ x in EPISIGN_OF_INTEREST for x in pcs_df.index ]
if osp.isfile(TRUE_CLUST_FILE):
    color_selected = EPISIGN_OF_INTEREST  # If known_cluster

x_compon = 'compon0'
y_compon = 'compon1'

fig = px.scatter(
        pcs_df[[x_compon,y_compon]],
        x=x_compon,
        y=y_compon,
        hover_data=[pcs_df.index],
        color=color_selected,
        labels={x_compon:':'.join([x_compon,"", dict_compon[x_compon]]), y_compon:':'.join([y_compon,"",dict_compon[y_compon]])}
)
fig.show()

In [ ]:
# Plot 2nd and 3rd compon from PCA
color_selected = [ x in EPISIGN_OF_INTEREST for x in pcs_df.index ]
#color_selected = EPISIGN_OF_INTEREST  # If known_cluster

x_compon = 'compon2'
y_compon = 'compon1'

fig = px.scatter(
        pcs_df[[x_compon,y_compon]],
        x=x_compon,
        y=y_compon,
        hover_data=[pcs_df.index],
        color=color_selected,
        labels={x_compon:':'.join([x_compon,"", dict_compon[x_compon]]), y_compon:':'.join([y_compon,"",dict_compon[y_compon]])}
)
fig.show()

#### Loadings

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=X.index,
    columns=[
        f"PC{i}"
        for i in range(pca.n_components_)
    ]
)
#print(loadings[['PC0','PC1']].head())

# Top N features of 2 first components
fig = px.bar(loadings.PC0.sort_values(key=lambda x:abs(x), ascending=False)[0:20])
fig.show()
fig2 = px.bar(loadings.PC1.sort_values(key=lambda x:abs(x), ascending=False)[0:20])
fig2.show()

### UMAP-like

In [ ]:
# MEMO: Scikit-learn does NOT support UMAP
#       + Not planned to add it (2024): https://github.com/scikit-learn/scikit-learn/issues/19393
isomap = Isomap(n_components=2, n_neighbors=2).fit_transform(to_PCA)
# SpectralEmbedding:
#isomap = SpectralEmbedding(n_components=2).fit_transform(to_PCA)
print(f"Trustworthiness of the low-dimensional embedding for UMAP-like: {trustworthiness(to_PCA, isomap, n_neighbors=2)}")

In [ ]:
# Put results into a dataframe
# WARN: bellow 'dict.keys' assume orderedDict
iso_df = pd.DataFrame(
    isomap,
    index=X.columns,
    columns=('compon0', 'compon1')
)
#print(iso_df.loc[EPISIGN_OF_INTEREST])

In [ ]:
color_selected = [ x in EPISIGN_OF_INTEREST for x in pcs_df.index ]
if osp.isfile(TRUE_CLUST_FILE):
    color_selected = EPISIGN_OF_INTEREST  # If known_cluster

fig_iso = plot_df(iso_df, color_selected)
fig_iso.show()

print("Isomap and SpectralEmbedding gives shit results")

### UMAP (true)

In [ ]:
# MEMO: Joris (methyl) uses 'n_neighbors=2'
#       But for RNA-seq, Copilot advice [5, 10, 15, 20, 30] for 30 to 100 samples
res_umap = umap.UMAP(n_neighbors=3).fit_transform(to_PCA)
print(f"Trustworthiness of the low-dimensional embedding for UMAP: {trustworthiness(to_PCA, res_umap, n_neighbors=2)}")

# Put results into a dataframe
# WARN: bellow 'dict.keys' assume orderedDict
umap_df = pd.DataFrame(
    res_umap,
    index=X.columns,
    columns=('compon0', 'compon1')
)
#print(umap_df.loc[EPISIGN_OF_INTEREST])

# Plot UMAP:
color_selected = [ x in EPISIGN_OF_INTEREST for x in pcs_df.index ]
if osp.isfile(TRUE_CLUST_FILE):
    color_selected = EPISIGN_OF_INTEREST  # If known_cluster

plot_df(umap_df, color_selected)

### Heatmaps

### t-SNE

In [ ]:
# Run t-SNE:
# MEMOs:
# - In Joris' paper they use 'preplex=2'
# - t-SNE is stochastic -> re-run multiple times ?
#
tsne = TSNE(n_components=2, perplexity=2).fit_transform(to_PCA)
print(f"Trustworthiness of the low-dimensional embedding for t-SNE: {trustworthiness(to_PCA, tsne, n_neighbors=2)}")

In [ ]:
# Put results into a dataframe
# WARN: bellow 'dict.keys' assume orderedDict
tsne_df = pd.DataFrame(
    tsne,
    index=X.columns,
    columns=('compon0', 'compon1')
)
#print(tsne_df.loc[EPISIGN_OF_INTEREST])

In [ ]:
# Plot t-SNE
color_selected = [ x in EPISIGN_OF_INTEREST for x in pcs_df.index ]
if osp.isfile(TRUE_CLUST_FILE):
    color_selected = EPISIGN_OF_INTEREST  # If known_cluster

fig_tsne = plot_df(tsne_df, color_selected)
fig_tsne.show()

## Clustering

### K-means

In [ ]:
# Select optimal number of cluster using 'silouhette' method
sil_scores = {}
inertias = {}

for k in range(2, 20):

    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=50
    )
    km_res = km.fit(pcs)

    labels = km_res.labels_
    inertias[k] = km_res.inertia_  # Intertia= Within-Cluster Sum of Squares

    sil = silhouette_score(
        pcs,
        labels
    )

    sil_scores[k] = sil


fig = px.line(
    x=inertias.keys(),
    y=inertias.values(),
)
fig.show()


best_k = max(
    sil_scores,
    key=sil_scores.get
)

print(
    "\nNombre optimal de clusters par silouhette:",
    best_k,
    f"(score: {sil_scores[best_k]})",
)


# Override NB_CLUST
best_k = 3

In [ ]:
# Clustering final avec le 'best nb cluster':
kmeans = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=50
)

clusters_kmeans = kmeans.fit_predict(pcs)

clusters_kmeans_df = pd.DataFrame(
    {
        "sample": X.T.index,
        "cluster": clusters_kmeans + 1
    }
)

print(clusters_kmeans_df.groupby('cluster').count())
clusters_kmeans_df.to_csv('clusters_kmeans.tsv', sep="\t", index=False)

### h-Clust

In [ ]:
# Seaborn's clustermap does h-clust + visu:
list_colors = ['blue','pink','red','green','gold','purple']
dict_colors = { i+1:c for i,c in enumerate(list_colors) }
color_interest_sample = ['pink' if s in EPISIGN_OF_INTEREST else 'blue' for s in to_PCA.index ]
#color_interest_sample = [ dict_colors[s] for s in EPISIGN_OF_INTEREST ]  If known_cluster

sns.set(font_scale=0.7)
hclust = sns.clustermap(
    to_PCA,
    metric='correlation',
    method='average',
    figsize=(20, 20),
    row_colors=color_interest_sample,
)

In [ ]:
# Recup des clusters du h-clust
from scipy.cluster.hierarchy import fcluster

# Les ech sont dans les lignes:
linkage = hclust.dendrogram_row.linkage

NB_CLUST = best_k
clusters_hclust = fcluster(
    linkage,
    t=NB_CLUST,
    criterion="maxclust"
)

# ALT: Decouper selon une dist plutot qu'un nombre de clust
fcluster(
    linkage,
    t=0.7,
    criterion="distance"
)

clusters_hclust_df = pd.DataFrame(
    {
        "sample": X.T.index,
        "cluster": clusters_hclust
    }
)
#print(clusters_hclust_df.groupby('cluster').count())
#clusters_hclust_df.to_csv('clusters_hclust.tsv', sep="\t", index=False)

### hClust on PCA res (with method=ward)

In [ ]:
# Seaborn's clustermap does h-clust + visu:
hclust_pca = sns.clustermap(
    pcs_df,
    method='ward',
    figsize=(20, 20),
    row_colors=color_interest_sample,
)

# Recup des clusters du h-clust
# Les ech sont dans les lignes:
linkage_pca = hclust_pca.dendrogram_row.linkage

NB_CLUST = best_k
clusters_hclust_pca = fcluster(
    linkage_pca,
    t=NB_CLUST,
    criterion="maxclust"
)

clusters_hclust_pca_df = pd.DataFrame(
    {
        "sample": X.T.index,
        "cluster": clusters_hclust_pca
    }
)
print(clusters_hclust_pca_df.groupby('cluster').count())
clusters_hclust_pca_df.to_csv('clusters_hclust_pca.tsv', sep="\t", index=False)

In [ ]:
### HDBSCAN
### TODO -> Dispo sklearn mais plusieurs hyperparam...
### Doc: https://hdbscan.readthedocs.io/en/latest/parameter_selection.html

### Bilan clustering

In [ ]:
# Compare predicted clusters between methods with ARI
# * 1.0 => clustering identique
# * 0.8+ => très forte concordance
# * 0.5 => concordance modérée
# * 0 => accord aléatoire
# * <0 => pire que le hasard

from sklearn.metrics import adjusted_rand_score

ari = adjusted_rand_score(
    clusters_kmeans,
    clusters_hclust_pca
)
print(f"Adjusted Rand Score hClust (PCA) vs k-means: {ari}")

ari2 = adjusted_rand_score(
    clusters_hclust,
    clusters_hclust_pca
)
print(f"Adjusted Rand Score hClust (PCA) vs hClust: {ari2}")

ari3 = adjusted_rand_score(
    clusters_hclust,
    clusters_kmeans
)
print(f"Adjusted Rand Score k-means vs hClust: {ari3}")

In [ ]:
# If we know true clusters:
if osp.isfile("known_clusters.tsv"):
    ari_kmeans = adjusted_rand_score(
        clusters_kmeans,
        known_clusters.cluster
    )
    print(f"Adjusted Rand Score k-means vs TRUTH: {ari_kmeans}")
    
    ari_hclust = adjusted_rand_score(
        clusters_hclust_pca,
        known_clusters.cluster
    )
    print(f"Adjusted Rand Score hClust (PCA) vs TRUTH: {ari_hclust}")

In [ ]:
# Plot PCA, but colored by clusters

x_compon = 'compon0'
y_compon = 'compon1'

fig = px.scatter(
        pcs_df,
        x=x_compon,
        y=y_compon,
        hover_data=[pcs_df.index],
        color=clusters_hclust_pca,
        labels={x_compon:':'.join([x_compon,"", dict_compon[x_compon]]), y_compon:':'.join([y_compon,"",dict_compon[y_compon]])}
)
fig.show()

## Classif

In [ ]:
# Disable classif is we do not have true labels:
assert(osp.isfile(TRUE_CLUST_FILE))

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import label_binarize

from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold, LeaveOneOut
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict

from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay, roc_auc_score
from sklearn.inspection import permutation_importance

from sklearn.multiclass import OneVsRestClassifier

In [ ]:
y = clusters_kmeans  # To use clusters predicted by k-means
y = known_clusters.cluster
X = to_PCA
print(known_clusters.drop_duplicates(subset=['cluster'], keep='first'))

FRACTION_DATASET = 0.2
#FRACTION_DATASET = float(sys.argv[1])  # BENCH

# Default = use full dataset (no split test/train):
X_used = X
y_used = y

### Split train/test

In [ ]:
if FRACTION_DATASET != 1:
    # MEMO: Choice of "test" is DETERMINISTIC here (always same list)
    X_train, X_test, y_train, y_test = train_test_split(
        to_PCA,
        y,
        test_size=FRACTION_DATASET,
        stratify=y,
        random_state=42
    )
    
    print(
        "Groups repartition in TRAIN:",
        y_train.value_counts()
    )
    print(
        "Groups repartition in TEST:",
        y_test.value_counts()
    )
    print(
        "Samples taken for TEST:",
        sorted(X_test.index)
    )
    print(f"Same samples: {'MEFV_CSG262318' in X_test.index}")

    X_used = X_train
    y_used = y_train

In [ ]:
# Gene selection: 200 a 1000 suffisent
# Mais si cohorte plus petite, plutot 20 ; 50 ou 100
BEST_GENES = 1000
#BEST_GENES = int(sys.argv[2])  # BENCH

# MEMO: Do it 1 common time here, to get list of 'best genes' for plots:
#       But already done by each 'Pipeline' call
best_genes_mask = SelectKBest(f_classif, k=BEST_GENES).fit(X_used, y_used).get_support()
best_genes_list = list(X.columns[best_genes_mask])

# Pour valid croisee repetee
cv_repet = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=100,
    random_state=42
)

In [ ]:
# ALT: Select genes using LASSO

# MEMO: LogisticRegressionCV and 'cv=5', could be used to auto-select 'C' hyperparam
#       -> But takes longer

lasso = LogisticRegression(
    l1_ratio=1,
    solver="saga",
    C=0.1,
    class_weight="balanced",
    random_state=42,
)

lasso.fit(X_used, y_used)

In [ ]:
# Get genes selected by LASSO 

selected_lasso = np.any(
    lasso.coef_ != 0,
    axis=0
)

selected_lasso_genes = X_used.columns[selected_lasso]

print(
    "Nombre de gènes retenus :",
    len(selected_lasso_genes)
)

# Voir les gènes les plus importants
importance_lasso = np.max(
    np.abs(lasso.coef_),
    axis=0
)

ranking_lasso = (
    pd.DataFrame({
        "gene": X_used.columns,
        "importance": importance_lasso
    })
    .sort_values(
        "importance",
        ascending=False
    )
)

print(
    ranking_lasso.head(50)
)

### Set Pipelines and fit

In [ ]:
# Random forest
# Avantages :
# - peu sensible aux paramètres ;
# - capte les interactions non linéaires ;
# - fournit une importance des gènes.

pipe_rf = Pipeline([
    ("select", SelectKBest(f_classif, k=BEST_GENES)),
    ("rf", RandomForestClassifier(n_estimators=1000, random_state=42))
])
pipe_rf.fit(
    X_used,
    y_used
)

### One-versus-rest approach

In [ ]:
# =========================================
# X : matrice d'expression ou PCs
# y : labels des classes
# =========================================

classes = np.unique(y_used)

# binarisation pour le calcul One-vs-Rest
y_bin = label_binarize(
    y_used,
    classes=classes
)

In [ ]:
# SVM
# Bonne option si nombre de gènes >> nombre d'échantillons (eg: 5000 genes vs 50 ech)

ONE_VS_REST = True
#ONE_VS_REST = True if sys.argv[3]=='True' else False  # BENCH

## Classic approach
pipe_svm = Pipeline([
    ("select", SelectKBest(f_classif, k=BEST_GENES)),
    ("svm", SVC(kernel="linear", class_weight="balanced", random_state=42))
])


## One-versus-rest approach
if ONE_VS_REST:
    # pipeline
    pipe_svm = Pipeline([
        (
            "select",
            SelectKBest(
                f_classif,
                k=BEST_GENES
            )
        ),
        (
            "svm",
            OneVsRestClassifier(
                SVC(
                    kernel="linear",
                    class_weight="balanced",
                    random_state=42,
                )
            )
        )
    ])
    
    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )
    
    # prédictions probabilistes obtenues
    # uniquement sur les folds de validation
    y_score = cross_val_predict(
        pipe_svm,
        X_used,
        y_used,
        cv=cv,
        method="decision_function",
    )
    
    # =========================================
    # AUC par classe
    # =========================================
    
    auc_results = {}
    
    for i, cls in enumerate(classes):
    
        auc = roc_auc_score(
            y_bin[:, i],
            y_score[:, i]
        )
    
        auc_results[cls] = auc
    
        print(
            f"Classe {cls}: "
            f"AUC = {auc:.3f}"
        )
    
    # =========================================
    # Macro AUC
    # =========================================
    
    macro_auc = roc_auc_score(
        y_bin,
        y_score,
        average="macro",
        multi_class="ovr"
    )
    
    print(
        f"\nMacro AUC = "
        f"{macro_auc:.3f}"
    )


# In any case, fit:
pipe_svm.fit(
    X_used,
    y_used
)

### Validation croisée répétée

In [ ]:
if False:

    # Random forest
    
    scores_rf = cross_val_score(
        pipe_rf,
        X_used,
        y_used,
        cv=cv_repet,
        scoring="accuracy"
    )
    
    print(scores_rf.mean())
    print(scores_rf.std())
    print(np.percentile(scores_rf, [5, 50, 95]))

In [ ]:
# SVM

scores_svm = cross_val_score(
    pipe_svm,
    X,
    y,
    cv=cv_repet,
    scoring="accuracy"
)

print(
    "FINAL_SVM_SCORE:",
    scores_svm.mean(),
    "SD=" + str(scores_svm.std())
)
print(scores_svm.std())
print(np.percentile(scores_svm, [5, 50, 95]))

### Features extraction

In [ ]:
# Random forest

forest = pipe_rf.named_steps["rf"]

# Genes predictifs par "mean decrease in impurity" (MDI):
# WARN: MDI biased towards high cardinality numerical variable (as many unique values as records).
rf_importances = pd.Series(forest.feature_importances_, index=best_genes_list).sort_values(ascending=False)
rf_std = np.std([tree.feature_importances_ for tree in forest.estimators_], axis=0)

#https://sklearn.org/stable/auto_examples/inspection/plot_permutation_importance.html#google_vignette

# Genes predictifs par "feature permutation":
# -> TOO INTENSIVE ???
#rf_permut = permutation_importance(
#    forest, X_test.T[best_genes_mask].T, y_test, n_repeats=10, random_state=42, n_jobs=2
#)
#rf_importances = pd.Series(rf_permut.importances_mean, index=best_genes_list)

#fig = px.bar(rf_importances, error_y=rf_std)
fig = px.bar(rf_importances[0:20])
fig.show()

In [ ]:
# SVM

# Genes predictifs par "mean decrease in impurity" (MDI):
# WARN: MDI biased towards high cardinality numerical variable (as many unique values as records).
# WARN: '.coeff_' works only with 'linear' kernel SVM
#       It is of shape (n_classes * (n_classes - 1) / 2, n_features)

svm_res = pipe_svm.named_steps["svm"]

if hasattr(svm_res, 'estimators_'):
    # If OneVsRest, multiple classifiers -> each one has its own genes of importance
    for cls, est in zip(
        svm_res.classes_,
        svm_res.estimators_
    ):
        svm_importances = pd.Series(abs(est.coef_[0]), index=best_genes_list).sort_values(ascending=False)
        print(cls)
        print(svm_importances.head())
    
    # On remarque que la classe des UBA1 a des coeff de valeurs plus faibles que les autres classes
    # Ca peut confirmer le fait que les UBA1 sont plus faciles à séparer (pas besoin de grands coeff)
    # WARN: Comparer des valeurs de coeff n'a pas forcément beaucoup de sens non plus


else:
    svm_importances = pd.Series(abs(svm_res.coef_[0]), index=best_genes_list).sort_values(ascending=False)
    fig = px.bar(svm_importances[0:20])
    fig.show()

In [ ]:
# If dataset were NOT splitted in train/test -> STOP here
assert list(X_used.index) != list(X.index)

### Predict on TEST + confusionMatrix (only if split train/test)

In [ ]:
# Random forest

pred_rf = pipe_rf.predict(X_test)

# Matrice de confusion:
fig = ConfusionMatrixDisplay.from_predictions(
    y_test,
    pred_rf
)
fig.im_

# Accuracy
print(
    accuracy_score(
        y_test,
        pred_rf
    )
)

In [ ]:
# SVM

pred_svm = pipe_svm.predict(X_test)

# Matrice de confusion:
fig = ConfusionMatrixDisplay.from_predictions(
    y_test,
    pred_svm
)
fig.im_

print(
    accuracy_score(
        y_test,
        pred_svm
    )
)

NB_CLASS = 3
decid_svm = pd.DataFrame(pipe_svm.decision_function(X_test),index=X_test.index,columns=[i+1 for i in range(NB_CLASS)])
print(decid_svm)

In [ ]:
assert False

### Validation croisee

In [ ]:
# WARN: Uniquement sur le train

# OLD
cv = 5
# ALT 1
StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
# ALT 2
LeaveOneOut()

# SVM
scores_svm = cross_val_score(
    pipe_svm,
    X_train,
    y_train,
    cv=cv
)
print("SVM cross_val scores:",
    scores_svm,
    "mean:", scores_svm.mean()
)

# RF
scores_rf = cross_val_score(
    pipe_rf,
    X_train,
    y_train,
    cv=cv
)
print("RF cross_val scores:",
    scores_rf,
    "mean:", scores_rf.mean()
)